# iLASH IBD segments in the final set: the shared phased HapMap3 extract (phased_hm3), cM from the Beagle GRCh38 map, one run per chromosome

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

## Functions

In [ ]:
def hexbin_safe(ax, x, y, **kw):
    """Pair density; cells with fewer than 21 pairs are not drawn."""
    return ax.hexbin(x, y, **{"gridsize": 60, "bins": "log", "cmap": "Greys", **kw, "mincnt": 21})

def tri_pairs(i, j):
    """Index of pair (i, j) in a lower triangle with diagonal, row by row (GCTA order)."""
    i, j = np.maximum(i, j), np.minimum(i, j)
    return i * (i + 1) // 2 + j

def genetic_map(c):
    """(bp, cM) for one chromosome; None: 1 cM per Mb (simulated data)."""
    if MAP_DIR is None:
        return None
    m = pd.read_csv(f"{MAP_DIR}/plink.chr{c}.GRCh38.map", sep=r"\s+", header=None, names=["chr", "id", "cm", "bp"])
    return m["bp"].to_numpy(), m["cm"].to_numpy()

def chrom_input(c, keep, hm3, work):
    """Phased set x HapMap3 for one chromosome -> iLASH .ped/.map; returns counts."""
    src = target_items(cfg, "phased_hm3")[c - 1][0][:-5]          # shared extract, everyone
    if src.startswith("gs://"):
        subprocess.run(["gcloud", "storage", "cp", f"{src}.pgen", f"{src}.pvar", f"{src}.psam", f"{work}/"], check=True,
                       capture_output=True)
        src = f"{work}/{os.path.basename(src)}"
    subprocess.run([cfg.tool("plink2"), "--pfile", src, "--keep", keep, "--maf", str(MIN_MAF), "--threads", "2",
                    "--silent", "--export", "vcf", "id-paste=iid", "--out", f"{work}/chr{c}.set"], check=True)
    if src.startswith(work):
        for e in ("pgen", "pvar", "psam"):
            os.remove(f"{src}.{e}")
    ids = hm3[hm3["chr"] == str(c)]
    want = set(ids["id"])
    cmd = f"bcftools query -H -f '%POS\\t%REF\\t%ALT[\\t%GT]\\n' {work}/chr{c}.set.vcf"
    p = subprocess.Popen(["bash", "-c", "set -o pipefail; " + cmd], stdout=subprocess.PIPE, text=True, bufsize=1 << 20)
    header = p.stdout.readline().rstrip("\n").split("\t")
    samples = [h.split("]", 1)[1].rsplit(":", 1)[0] for h in header[3:]]
    pos, ref, alt, haps = [], [], [], []
    n_read = n_missing = 0
    for line in p.stdout:
        f = line.rstrip("\n").split("\t")
        n_read += 1
        if f"{c}:{f[0]}:{f[1]}:{f[2]}" not in want:
            continue
        g = np.array(f[3:], dtype="U3").view("U1").reshape(-1, 3)   # allele | allele
        ok = (g[:, 1] == "|") & np.isin(g[:, 0], ["0", "1"]) & np.isin(g[:, 2], ["0", "1"])
        if (~ok).mean() > 0.01:   # more than 1% missing or unphased: drop the SNP
            n_missing += 1; continue
        h = np.stack([(g[:, 0] == "1") & ok, (g[:, 2] == "1") & ok])   # the rest set to REF
        maf = h.mean()
        if min(maf, 1 - maf) < MIN_MAF:
            continue
        pos.append(int(f[0])); ref.append(f[1]); alt.append(f[2]); haps.append(h)
    if p.wait() != 0:
        raise RuntimeError(f"bcftools failed on chr{c}")
    os.remove(f"{work}/chr{c}.set.vcf")
    pos = np.array(pos)
    if not len(pos):
        return {"chr": c, "sites_read": n_read, "snps": 0}
    gm = genetic_map(c)
    cm = pos / 1e6 if gm is None else np.interp(pos, gm[0], gm[1])
    pd.DataFrame({"chr": c, "id": [f"{c}:{a}:{b}:{d}" for a, b, d in zip(pos, ref, alt)], "cm": cm, "bp": pos}) \
      .to_csv(f"{work}/chr{c}.map", sep="\t", header=False, index=False)
    H = np.stack(haps, axis=1)                       # 2 x snps x samples
    letters = np.array([ref, alt]).astype("S1")      # 2 x snps
    with open(f"{work}/chr{c}.ped", "wb") as fh:
        for j, s in enumerate(samples):
            row = np.empty((len(pos), 4), dtype="S1")
            row[:, 0] = np.where(H[0, :, j], letters[1], letters[0])
            row[:, 1] = b" "
            row[:, 2] = np.where(H[1, :, j], letters[1], letters[0])
            row[:, 3] = b" "
            fh.write(f"{s} {s} 0 0 0 -9 ".encode() + row.tobytes()[:-1] + b"\n")
    return {"chr": c, "sites_read": n_read, "snps": len(pos), "dropped_missing": n_missing,
            "cm_span": round(float(cm.max() - cm.min()), 1), "samples": len(samples)}

## Inputs: phased set x HapMap3, per chromosome

In [ ]:
import time
STAGE = "ilash"
CHRS = [int(c) for c in os.environ.get("ILASH_CHRS", "").split(",") if c] or list(range(1, 23))
TEST = len(CHRS) < 22   # ILASH_CHRS=21,22: only those chromosomes, a separate output, stage not marked done
if not TEST:
    require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
if TEST:
    OUT = OUT.replace("pairs.tsv", f"pairs.chr{'_'.join(map(str, CHRS))}.tsv")
    missing = [c for c in CHRS if not get_store(target_items(cfg, "phased_hm3")[c - 1][0]).exists(target_items(cfg, "phased_hm3")[c - 1][0])]
    assert not missing, f"phased_hm3 missing for chr {missing}"
    rep.note(test_chromosomes=",".join(map(str, CHRS)))
RUN = needs_run(OUT, FORCE)
WORK = os.path.join(os.path.dirname(OUT), "work")
KEEP = single_target(cfg, "ancestry_round2")
MIN_MAF = cfg.param("qc_maf")
ILASH = cfg.tool("ilash")
MAP_DIR = None
if not cfg.params.get("simulated"):
    MAP_DIR = os.path.join(os.path.expanduser(cfg.params.get("scratch", "~")), "genetic_map_grch38")
    if not os.path.exists(f"{MAP_DIR}/plink.chr1.GRCh38.map"):
        os.makedirs(MAP_DIR, exist_ok=True)
        subprocess.run(["bash", "-c", f"curl -fsSL -o {MAP_DIR}/m.zip '{cfg.input('genetic_map')}' && "
                        f"unzip -o -q -j {MAP_DIR}/m.zip 'no_chr_in_chrom_field/*' -d {MAP_DIR}"], check=True)
hm3 = pd.read_csv(cfg.input("hm3_ids"), header=None, names=["id"])
hm3[["chr", "pos"]] = hm3["id"].str.split(":", expand=True)[[0, 1]]
hm3["chrom"] = "chr" + hm3["chr"]
print(len(hm3), "HapMap3 IDs;", sum(1 for _ in open(KEEP)), "people in the final set")

In [ ]:
if RUN:
    from concurrent.futures import ThreadPoolExecutor
    os.makedirs(WORK, exist_ok=True)
    with ThreadPoolExecutor(max_workers=min(6, THREADS)) as pool:
        counts = pd.DataFrame(list(pool.map(lambda c: chrom_input(c, KEEP, hm3, WORK), CHRS)))
    counts.to_csv(f"{WORK}/counts.tsv", sep="\t", index=False)
counts = pd.read_csv(f"{WORK}/counts.tsv", sep="\t")
rep.table(counts.drop(columns=["samples"], errors="ignore"), "snps_by_chr")
rep.check("every chromosome has SNPs", (counts["snps"] > 0).all())
counts

## iLASH (built from github.com/roohy/iLASH on first use; parameters from its sample_config)

In [ ]:
if RUN and not os.path.exists(ILASH):
    subprocess.run(["bash", "-c", f"""set -e; src=$(mktemp -d); git clone -q --depth 1 https://github.com/roohy/iLASH.git $src
        mkdir -p $src/build && cd $src/build && cmake .. -DCMAKE_BUILD_TYPE=Release >/dev/null && make -j{THREADS} >/dev/null
        mkdir -p $(dirname {ILASH}) && cp ilash {ILASH}"""], check=True)
PARAMS = {"slice_size": 350, "step_size": 350, "perm_count": 20, "shingle_size": 15, "shingle_overlap": 0,
          "bucket_count": 5, "max_thread": THREADS, "match_threshold": 0.99, "interest_threshold": 0.70,
          "min_length": 2.9, "auto_slice": 1, "slice_length": 2.9, "cm_overlap": 1, "minhash_threshold": 55,
          "seed": cfg.param("seed")}
if RUN:
    for c in counts.loc[counts["snps"] > 0, "chr"]:
        conf = f"{WORK}/chr{c}.config"
        with open(conf, "w") as fh:
            fh.write(f"map {WORK}/chr{c}.map\nped {WORK}/chr{c}.ped\noutput {WORK}/chr{c}.match\n")
            fh.write("".join(f"{k} {v}\n" for k, v in PARAMS.items()))
        subprocess.run([ILASH, conf], check=True, stdout=subprocess.DEVNULL)
        print(f"chr{c} done", time.strftime("%T"), flush=True)

## Pairs: total shared cM and an IBD kinship

In [ ]:
if RUN:
    seg = pd.concat([pd.read_csv(f"{WORK}/chr{c}.match", sep="\t", header=None,
                                 names=["f1", "h1", "f2", "h2", "chr", "start", "end", "snp1", "snp2", "cm", "sim"])
                     for c in counts.loc[counts["snps"] > 0, "chr"] if os.path.getsize(f"{WORK}/chr{c}.match")])
    seg["i1"], seg["i2"] = seg["h1"].str.rsplit("_", n=1).str[0], seg["h2"].str.rsplit("_", n=1).str[0]
    seg = seg[seg["i1"] != seg["i2"]]                 # matches within a person (autozygosity) left out
    a, b = np.minimum(seg["i1"], seg["i2"]), np.maximum(seg["i1"], seg["i2"])
    L = float(counts["cm_span"].sum())
    pairs = (seg.assign(id1=a, id2=b).groupby(["id1", "id2"])["cm"].agg(n_segments="size", total_cm="sum").reset_index())
    pairs["kinship_ibd"] = pairs["total_cm"] / (4 * L)   # haplotype-pair sharing over 4 genome lengths
    if TEST:
        pairs.to_csv(OUT, sep="\t", index=False)
    else:
        with record(cfg, STAGE, OUT) as meta:
            pairs.to_csv(OUT, sep="\t", index=False)
            meta.update(n_pairs=len(pairs), genome_cm=L, params=PARAMS)
pairs = pd.read_csv(OUT, sep="\t", dtype={"id1": str, "id2": str})
n = sum(1 for _ in open(KEEP))
rep.count("n_people", n); rep.count("n_pairs_with_segments", len(pairs))
rep.note(genome_cm=float(counts["cm_span"].sum()), min_segment_cm=PARAMS["min_length"])

In [ ]:
edges = [2.9, 10, 20, 50, 100, 200, 500, 1000, 2000, 10000]
pairs["bin"] = pd.cut(pairs["total_cm"], edges)
tab = pairs.groupby("bin", observed=False).agg(pairs=("total_cm", "size")).reset_index()
cmp = {}
for stage, path_fn in [("grm", lambda: target_items(cfg, "grm_shards")), ("pcrelate", lambda: [(single_target(cfg, "pcrelate"), None)])]:
    try:
        uris = [u for u, _ in path_fn()]
        if not all(os.path.exists(u) for u in uris):
            continue
        id_file = uris[0].split(".grm.bin.")[0] + ".grm.id" if stage == "grm" else uris[0].replace(".bin", ".id")
        ids = pd.read_csv(id_file, sep="\t", header=None, dtype=str).iloc[:, -1]
        tri = np.concatenate([np.fromfile(u, dtype="float32") for u in uris])
        pos = pd.Series(np.arange(len(ids)), index=ids)
        ok = pairs["id1"].isin(pos.index) & pairs["id2"].isin(pos.index)
        v = np.full(len(pairs), np.nan)
        v[ok] = tri[tri_pairs(pos[pairs.loc[ok, "id1"]].to_numpy(), pos[pairs.loc[ok, "id2"]].to_numpy())]
        cmp[stage] = v / 2 if stage == "grm" else v   # both as kinship
        pairs[f"{stage}_kinship"] = cmp[stage]
        g = pairs.groupby("bin", observed=False)[f"{stage}_kinship"]
        tab[f"mean_{stage}_kinship"] = g.mean().round(4).values
    except KeyError:
        pass
tab["mean_ibd_kinship"] = pairs.groupby("bin", observed=False)["kinship_ibd"].mean().round(4).values
small = tab["pairs"].between(1, 20)
for col in tab.columns[2:]:
    tab[col] = tab[col].mask(small)
tab["bin"] = tab["bin"].astype(str)
shown = rep.table(tab, "pairs_by_shared_cm", counts=["pairs"])
display(shown)

fig, axes = plt.subplots(1, 1 + len(cmp), figsize=(5.5 * (1 + len(cmp)), 4.5), squeeze=False)
h, e = np.histogram(np.log10(pairs["total_cm"]), bins=60)
axes[0, 0].stairs(np.where(h >= 21, h, 0), e)
axes[0, 0].set(yscale="log", xlabel="log10 total shared cM per pair", ylabel="pairs", title="bins with 21+ pairs")
for ax, (stage, v) in zip(axes[0, 1:], cmp.items()):
    m = ~np.isnan(v)
    hexbin_safe(ax, pairs.loc[m, "kinship_ibd"], v[m])
    ax.plot([0, 0.5], [0, 0.5], "r:", lw=0.8)
    ax.set(xlabel="IBD kinship (iLASH)", ylabel=f"{stage} kinship", title="pairs with segments")
    if m.sum() >= 21:   # no statistic from fewer than 21 pairs
        rep.note(**{f"corr_ibd_{stage}": round(float(np.corrcoef(pairs.loc[m, "kinship_ibd"], v[m])[0, 1]), 3)})
plt.tight_layout()
rep.fig(fig, "ilash_vs_kinship", "pairs with at least one segment of 2.9+ cM; hexagons and bins with 21+ pairs"); plt.show()
rep.save()